In [ ]:
# -*- coding: utf-8 -*-
"""
试卷 B - 基于 SVM 的鸢尾花数据分类模型构建与评估
要求：数据加载与分析、多模型交叉验证、SVM 训练与评估、绘制箱线图/热力图/决策边界
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

pandas：用于数据处理和保存 CSV 文件。

numpy：数值计算，数组操作。

matplotlib.pyplot：绘图基础库。

seaborn：基于 matplotlib 的高级统计绘图，用于绘制混淆矩阵热力图。

load_iris：加载鸢尾花数据集的工具函数。

train_test_split：划分训练集和测试集。

cross_val_score：计算交叉验证得分。

StratifiedKFold：分层 K 折交叉验证器，保持每折中各类别比例与原始数据一致。

LogisticRegression：逻辑回归模型（用于对比）。

SVC：支持向量机分类器。

accuracy_score：计算分类准确率。

confusion_matrix：生成混淆矩阵。

In [ ]:
# 设置 matplotlib 中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

设置无衬线字体为黑体（SimHei），以支持图表中的中文显示。

解决负号显示异常的问题（将负号正常显示）。

In [ ]:
# ==================== 1. 数据集准备与分析 ====================
# 加载鸢尾花数据集
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

load_iris()：返回一个类似字典的对象，包含数据、目标值、特征名、类别名等。

X：特征数据，形状 (150, 4)。

y：标签，0、1、2 分别代表 setosa、versicolor、virginica。

feature_names：['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']。

target_names：['setosa', 'versicolor', 'virginica']。

In [ ]:
# 保存为 CSV 文件
df = pd.DataFrame(X, columns=feature_names)
df['class'] = y
df.to_csv('iris.csv', index=False)
print("已保存 iris.csv")

将特征数据构建为 pandas DataFrame，列名为特征名。

添加一列 'class' 存储标签。

将 DataFrame 保存为 CSV 文件，不保存行索引。

打印提示信息。

In [ ]:
# 基本信息
print(f"数据集形状: {df.shape}")
print("\n前10行数据:")
print(df.head(10))
print("\n统计信息:")
print(df.describe())
print("\n各类别样本数量:")
print(df['class'].value_counts().sort_index())

df.shape：输出 (150, 5)。

head(10)：显示前10行数据。

describe()：计算数值列的基本统计量（均值、标准差、四分位数等）。

value_counts()：统计每个类别出现的次数，sort_index() 按类别0,1,2排序。

In [ ]:
# 绘制箱线图（按类别分组）
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, feature in enumerate(feature_names):
    ax = axes[i//2, i%2]
    df.boxplot(column=feature, by='class', ax=ax, grid=False)
    ax.set_title(feature)
    ax.set_xlabel('类别')
    ax.set_ylabel('数值')
plt.suptitle('各特征按类别分组的箱线图', size=16)
plt.tight_layout()
plt.show()

创建 2×2 的子图布局。

对每个特征（共4个），在对应的子图位置上绘制按类别分组的箱线图。

df.boxplot(column=feature, by='class', ax=ax)：以 class 为分组依据，绘制该特征的箱线图。

grid=False：不显示网格线。

设置子图标题、坐标轴标签。

添加总标题，自动调整子图间距，显示图形。

In [ ]:
# ==================== 2. 多模型交叉验证对比 ====================
# 划分训练集和测试集 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
print(f"\n训练集大小: {X_train.shape[0]}, 测试集大小: {X_test.shape[0]}")

test_size=0.2：20% 数据作为测试集，80% 作为训练集。

random_state=1：固定随机种子，保证结果可重复。

打印训练/测试样本数量。

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(solver='lbfgs', max_iter=200),
    'SVM (RBF)': SVC(gamma='auto')
}

定义两个要比较的模型：逻辑回归和 SVM（RBF 核）。

逻辑回归使用 LBFGS 求解器，最大迭代次数 200。

SVM 使用 gamma='auto'（即 1/n_features = 0.25）。

In [ ]:
results = []
names = []
for name, model in models.items():
    kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
    cv_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')
    results.append(cv_scores)
    names.append(name)
    print(f"{name}: 平均准确率={cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

对每个模型：

创建 10 折分层交叉验证器（shuffle 打乱，固定随机种子）。

cross_val_score：在训练集上执行 10 折交叉验证，评估指标为准确率。

将 10 个得分存入 results，模型名存入 names。

打印平均准确率和标准差。

In [ ]:
# 绘制算法对比箱线图
plt.figure(figsize=(8, 6))
plt.boxplot(results, labels=names)
plt.title('算法交叉验证准确率对比')
plt.ylabel('准确率')
plt.grid(axis='y')
plt.show()

创建新图形。

plt.boxplot(results, labels=names)：绘制两个模型的交叉验证得分箱线图。

添加标题、Y轴标签、网格线，显示图形。

In [ ]:
# ==================== 3. SVM 模型训练与评估 ====================
svm = SVC(gamma='auto')
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

创建 SVM 分类器（参数同上）。

使用训练集数据拟合模型。

对测试集进行预测，得到预测标签 y_pred。

In [ ]:
# 准确率
acc = accuracy_score(y_test, y_pred)
print(f"\nSVM 在测试集上的准确率: {acc:.4f}")


计算预测准确率并打印。

In [ ]:
# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, yticklabels=target_names)
plt.title('SVM 混淆矩阵热力图')
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.show()

confusion_matrix 生成混淆矩阵（3×3）。

使用 seaborn 的 heatmap 绘制热力图：

annot=True：在每个格子中显示数字。

fmt='d'：数字格式为整数。

cmap='Blues'：颜色映射为蓝色系。

xticklabels/yticklabels：坐标轴标签使用花的种类名称。

添加标题、轴标签，显示图形。

In [ ]:
# 决策边界图（使用花瓣长度和花瓣宽度两个特征）
X_2d = X[:, 2:4]  # 取 petal length, petal width
y_2d = y
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(X_2d, y_2d, test_size=0.2, random_state=1)
svm_2d = SVC(gamma='auto')
svm_2d.fit(X_train_2d, y_train_2d)

因为原始数据是4维，无法直接可视化决策边界，故只取后两个特征（花瓣长和宽）。

再次划分训练/测试集（保持一致性）。

在新的二维数据集上训练 SVM 模型。

In [ ]:
# 生成网格点
h = 0.02
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

确定特征空间的边界（扩大0.5范围）。

创建网格点坐标矩阵 xx, yy，步长 0.02。

对每个网格点进行预测，得到类别标签 Z。

将 Z 重塑为与 xx 相同的形状，用于绘制等高线。

In [ ]:
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y_2d, edgecolors='k', cmap=plt.cm.coolwarm)
plt.xlabel('花瓣长度 (cm)')
plt.ylabel('花瓣宽度 (cm)')
plt.title('SVM 决策边界（基于花瓣特征）')
plt.show()

print("\n所有分析完成。")

创建新图形。

plt.contourf：绘制填充等高线，展示决策区域，透明度0.3，颜色映射 coolwarm。

plt.scatter：绘制原始数据点，颜色 c=y_2d 按真实标签着色，边缘黑色。

添加坐标轴标签、标题，显示图形。